# MercadoTico 360: Simulación y carga de datos (CouchDB)

Generación y carga los datos del caso 2: **25 000 productos** (8 categorías con atributos variables), **10 000 clientes** y **50 000 pedidos**.

## Diseño de documentos (Requisito 1: qué se embebe y qué se referencia)

| Documento | `_id` | Estrategia | Justificación |
|---|---|---|---|
| `product` | `PROD-000001` | Documento independiente, `atributos` embebidos en el documento | Cada categoría tiene atributos distintos; viven dentro del producto sin esquema fijo (Req. 2). |
| `customer` | `CLI-000001` | Documento propio | Un cliente puede acumular numerosos pedidos; embeberlos dentro del documento del cliente produciría documentos cada vez más grandes y dificultaría su actualización. |
| `order` | `PED-000001` | `lineas_detalle` **embebidas** + `cliente_id` y `producto_id` **referenciados** | El pedido se consulta como una unidad junto con sus líneas de detalle. Cada línea guarda una copia histórica (*snapshot*) (nombre, categoría, precio) para reconstruir la compra aunque el producto cambie. |

Al utilizar identificadores `_id` definidos, una nueva ejecución no crea documentos duplicados; los conflictos con documentos existentes son identificados durante la carga.

In [ ]:
import os
import random
import sys
import json
from datetime import date, timedelta
from getpass import getpass

import requests

try:                                   # carga el .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ----------------------------------------------------------------------
# CONFIGURACIÓN
# Configuración mediante variables de entorno o entrada manual.
# Nunca guarde credenciales en el código ni en el repositorio.
# Ejemplo de .env.example:
# COUCHDB_HOST=localhost
# COUCHDB_PORT=5984
# COUCHDB_USER=admin
# COUCHDB_PASS=
# DATABASE_NAME=mercadotico360
# ----------------------------------------------------------------------

def leer_config(nombre, pregunta, defecto=None, secreto=False):
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto

COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

BASE_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}"
DB_URL   = f"{BASE_URL}/{DATABASE_NAME}"
BULK_URL = f"{DB_URL}/_bulk_docs"

SESSION = requests.Session()           # reutiliza la conexión (más rápido)
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)



# PARÁMETROS DE LA SIMULACIÓN
SEED           = 42        # misma semilla = mismos datos
NUM_PRODUCTOS  = 25_000    # cantidad de productos
NUM_CLIENTES   = 10_000    # cantidad declientes    
NUM_PEDIDOS    = 50_000    # cantidad de pedidos
NUM_CATEGORIAS = 8         # cantidad de categorías en el mercado
BATCH_SIZE     = 5_000     # carga en lotes de 5 mil documentos

rng = random.Random(SEED)

## Funciones de conexión y carga

In [ ]:
def verificar_o_crear_bd():
    # comprueba la conexión con CouchDB y crea la base de datos si no existe.
    try:
        res = SESSION.get(DB_URL, timeout=10)
        if res.status_code == 200:
            print(f"La base de datos '{DATABASE_NAME}' ya existe.")
        elif res.status_code == 404:
            creada = SESSION.put(DB_URL, timeout=10)
            if creada.status_code not in (201, 202):
                raise RuntimeError(f"No se pudo crear la BD: {creada.status_code} - {creada.text}")
            print(f"Base de datos '{DATABASE_NAME}' creada.")
        elif res.status_code == 401:
            raise RuntimeError("Usuario o contraseña incorrectos.")
        else:
            raise RuntimeError(f"Error al verificar la BD: {res.status_code} - {res.text}")
    except requests.exceptions.ConnectionError:
        raise RuntimeError(f"No hay conexión con CouchDB en {BASE_URL}. ¿Está el servicio encendido?")


def enviar_lote(documentos):
    # envía documentos a /_bulk_docs. Devuelve (insertados, ya_existian, errores).
    res = SESSION.post(BULK_URL, json={"docs": documentos}, timeout=120)
    res.raise_for_status()
    # se revisa cada resultado porque un lote puede contener documentos con error
    fallos = [r for r in res.json() if "error" in r]
    ya_existian = sum(1 for r in fallos if r["error"] == "conflict")
    return len(documentos) - len(fallos), ya_existian, len(fallos) - ya_existian


def cargar_documentos(documentos):
    # carga todos los documentos por lotes y muestra el progreso.
    total = len(documentos)
    insertados = existentes = errores = 0
    for inicio in range(0, total, BATCH_SIZE):
        lote = documentos[inicio:inicio + BATCH_SIZE]
        ok, dup, err = enviar_lote(lote)
        insertados, existentes, errores = insertados + ok, existentes + dup, errores + err
        print(f"  {min(inicio + BATCH_SIZE, total):>6}/{total} procesados")
    print(f"[RESUMEN] nuevos: {insertados} | existentes: {existentes} | con error: {errores}")

## Productos con atributos heterogéneos
Cada categoría define su propio conjunto de atributos dentro de CATALOGO. La estructura permite incorporar nuevas categorías agregando una nueva entrada al catálogo. Los precios sintéticos se expresan en colones costarricenses y se generan dentro de rangos específicos para cada categoría.

In [ ]:
CATALOGO = {
    "Alimentos": {
        "ejemplo": "Café",
        "rango_precio": (1000, 30000),
        "atributos": lambda: {
            "peso_g": rng.choice([250, 500, 1000, 2000]),
            "tipo": rng.choice(["Café", "Cereal", "Snack", "Bebida", "Conserva"]),
            "origen": rng.choice(["Costa Rica", "Colombia", "México", "Brasil", "Perú"]),
        },
    },
    "Ropa": {
        "ejemplo": "Camiseta",
        "rango_precio": (5000, 80000),
        "atributos": lambda: {
            "talla": rng.choice(["XS", "S", "M", "L", "XL"]),
            "color": rng.choice(["Rojo", "Azul", "Negro", "Blanco"]),
            "material": rng.choice(["Algodón", "Poliéster", "Cuero", "Lino"]),
        },
    },
    "Tecnologia": {
        "ejemplo": "Laptop",
        "rango_precio": (10000, 1500000),
        "atributos": lambda: {
            "marca": f"TechBrand-{rng.randint(1, 50)}",
            "ram_gb": rng.choice([4, 8, 16, 32, 64]),            # usado por idx_technology_ram
            "almacenamiento_gb": rng.choice([128, 256, 512, 1024, 2048]),
        },
    },
    "Hogar": {
        "ejemplo": "Lámpara",
        "rango_precio": (3000, 250000),
        "atributos": lambda: {
            "material": rng.choice(["Metal", "Madera", "Vidrio", "Plástico"]),
            "color": rng.choice(["Blanco", "Negro", "Dorado", "Plateado"]),
            "altura_cm": rng.choice([20, 35, 50, 80, 120, 160]),
        },
    },
    "Cuidado_Personal": {
        "ejemplo": "Shampoo",
        "rango_precio": (2000, 50000),
        "atributos": lambda: {
            "marca": f"CareBrand-{rng.randint(1, 50)}",
            "volumen_ml": rng.choice([100, 250, 400, 750, 1000]),
            "tipo_cabello": rng.choice(["Graso", "Seco", "Mixto", "Normal", "Rizado"]),
        },
    },
    "Libros": {
        "ejemplo": "Novela",
        "rango_precio": (4000, 40000),
        "atributos": lambda: {
            "autor": f"Autor {rng.randint(1, 500)}",
            "editorial": rng.choice(["Planeta", "Alfaguara", "Anagrama", "Santillana", "EUNED"]),
            "idioma": rng.choice(["Español", "Inglés", "Francés", "Portugués"]),
        },
    },
    "Mascotas": {
        "ejemplo": "Alimento para mascota",
        "rango_precio": (2000, 80000),
        "atributos": lambda: {
            "especie": rng.choice(["Perro", "Gato", "Ave", "Pez", "Roedor"]),
            "peso_kg": rng.choice([1, 2, 5, 10, 15, 20]),
            "sabor": rng.choice(["Pollo", "Res", "Pescado", "Cordero", "Mixto"]),
        },
    },
    "Artesanias": {
        "ejemplo": "Taza de cerámica",
        "rango_precio": (3000, 150000),
        "atributos": lambda: {
            "material": rng.choice(["Cerámica", "Madera", "Metal", "Textil", "Vidrio"]),
            "tecnica": rng.choice(["Torno", "Tallado", "Tejido", "Pintado a mano", "Soplado"]),
            "region_origen": rng.choice(["Guanacaste", "Sarchí", "Guaitil", "Chorotega", "San Vicente de Nicoya"]),
        },
    },
}


def generar_productos(cantidad=NUM_PRODUCTOS):
    categorias = list(CATALOGO)
    productos = []
    for i in range(1, cantidad + 1):
        cat = rng.choice(categorias)                             
        precio_min, precio_max = CATALOGO[cat]["rango_precio"]  
        producto_id = f"PROD-{i:06d}"
        productos.append({
            "_id": producto_id,
            "type": "product",
            "nombre": f"{CATALOGO[cat]['ejemplo']} {i}",
            "categoria": cat,
            "precio": rng.randint(precio_min, precio_max),
            "stock": rng.randint(0, 500),
            "atributos": CATALOGO[cat]["atributos"](),
        })
    return productos

## Pedidos con líneas de detalle anidadas
Cada línea **referencia** el producto (`producto_id`) y además guarda un **snapshot** de nombre, categoría y precio. Así el pedido se puede reconstruir aunque el producto cambie, y las agregaciones por categoría no necesitan consultar el catálogo.

In [ ]:
ESTADOS = ["completado", "pendiente", "cancelado"]


def generar_pedidos(productos, clientes, cantidad=NUM_PEDIDOS):
    # genera pedidos de 1 a 4 líneas, tomando los datos de productos reales.
    inicio = date(2026, 1, 1)
    pedidos = []
    for i in range(1, cantidad + 1):
        lineas = []
        for _ in range(rng.randint(1, 4)):
            prod = rng.choice(productos)
            unidades = rng.randint(1, 3)
            lineas.append({
                "producto_id": prod["_id"],          # referencia
                "nombre_snapshot": prod["nombre"],           # snapshot histórico
                "categoria_snapshot": prod["categoria"],
                "cantidad": unidades,
                "precio_unitario": prod["precio"],
                "subtotal": round(prod["precio"] * unidades, 2),
            })

        pedido_id = f"PED-{i:06d}"
        cliente = rng.choice(clientes)
        pedidos.append({
            "_id": pedido_id,
            "type": "order",
            "cliente_id": cliente["_id"],   # referencia
            "fecha_pedido": (inicio + timedelta(days=rng.randint(0, 270))).isoformat(),
            "lineas_detalle": lineas,                                  # embebido
            "monto_total": round(sum(l["subtotal"] for l in lineas), 2),
            "estado": rng.choice(ESTADOS),
        })
    return pedidos